# 모니터링과 알림

오류율과 p95 지연이 함께 기준을 넘은 상태가 두 구간 이어지는지 확인합니다.

In [1]:
from pathlib import Path
import json, subprocess, sys
lab = next((p for p in [Path.cwd(), *Path.cwd().parents] if p.name == "ch33-cicd" and (p / "pyproject.toml").is_file()), Path("labs/part7/ch33-cicd")).resolve()
rows = [json.loads(x) for x in (lab / "data/metrics.jsonl").read_text().splitlines()]
print("rows=", len(rows))

rows= 4


In [2]:
for row in rows:
    error_rate = row["errors"] / row["requests"]
    firing = error_rate > 0.05 and row["p95_ms"] > 800
    print(row["minute"], f"error_rate={error_rate:.3f}", f"p95={row['p95_ms']}", firing)

1 error_rate=0.010 p95=320 False
2 error_rate=0.017 p95=360 False
3 error_rate=0.082 p95=920 True
4 error_rate=0.092 p95=1100 True


In [3]:
result = subprocess.run([sys.executable, "scripts/check_metrics.py"], cwd=lab, text=True, capture_output=True)
print(result.stdout.strip())
print("exit_code=", result.returncode)
assert result.returncode == 1

1 0.01 320 False
2 0.017 360 False
3 0.082 920 True
4 0.092 1100 True
alert: FIRING longest_streak=2
exit_code= 1


## 결과 해석

1·2분에는 두 조건을 동시에 만족하지 않았습니다. 3·4분에는 오류율 5% 초과와 p95 800ms 초과가 함께 관찰되어 경보 상태가 되었습니다. 이 고정 자료는 임계값 동작만 보여 주며 실제 서비스 정상 범위를 정하지 않습니다.